In [3]:
import os

import kymnasium as kym
import gymnasium as gym


env = gym.make(
    'kymnasium/AvoidBlurp-Discrete-Ballistic-Normal-Stage-1',
    render_mode='none'
)
env.reset()

({'mario': array([288., 768., 336., 816.,   0.], dtype=float32),
  'blurps': array([[0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.

In [4]:
RANDOM_SEED = 42

REPLAY_BUFFER_SIZE = 100000
BATCH_SIZE = 64
ALPHA = 0.6
BETA = 0.4

EPSILON_MIN = 0.05
EPSILON_DECAY_STEPS = 200000
INIT_EXPLORATION = 10000

INTERVAL_UPDATE = 8
LEARNING_RATE = 0.00025
TAU = 0.005

GAMMA = 0.98

STATE_DIM = 3 + 30 * 5
STATE_SEQ = 8
ACTION_DIM = 3

N_MONITOR = 100
WIDTH, HEIGHT = 624, 912

In [5]:
import numpy as np


class ReplayBuffer:
    def __init__(
            self, capacity: int, alpha: float, beta: float, seed: int = None,
    ):
        self._alpha = alpha
        self._beta = beta
        self._capacity = capacity

        self._states = np.zeros(shape=(self._capacity, STATE_SEQ, STATE_DIM), dtype='float32')
        self._actions = np.zeros(shape=(self._capacity, ), dtype='int32')
        self._rewards = np.zeros(shape=(self._capacity,), dtype='float32')
        self._next_states = np.zeros(shape=(self._capacity, STATE_SEQ, STATE_DIM), dtype='float32')
        self._dones = np.zeros(shape=(self._capacity,), dtype='float32')
        self._priorities = np.ones(shape=(self._capacity,), dtype='float32')

        self._max_priority = 1.0
        self._size = 0
        self._index = 0

        self._random = np.random.default_rng(RANDOM_SEED)

    @property
    def size_(self):
        return self._size

    def add(self, state, action, reward, next_state, done):
        priority = self._priorities.max() if self._size > 0 else 1.0

        self._states[self._index] = state
        self._actions[self._index] = action
        self._rewards[self._index] = reward
        self._next_states[self._index] = next_state
        self._dones[self._index] = done
        self._priorities[self._index] = priority
        self._index = (self._index + 1) % self._capacity
        self._size = min(self._size + 1, self._capacity)

    def sample(self, batch_size):
        probs = self._priorities[:self._size] ** self._alpha
        probs /= probs.sum()

        indices = self._random.choice(self._size, size=batch_size, p=probs)

        weights = (self._size * probs[indices]) ** (-self._beta)
        weights /= weights.max()
        weights = np.array(weights, dtype=np.float32)

        return (
            self._states[indices],
            self._actions[indices],
            self._rewards[indices],
            self._next_states[indices],
            self._dones[indices],
            indices,
            weights
        )

    def update_priorities(self, indices, priorities):
        priorities = np.asarray(priorities, dtype=np.float32).reshape(-1)
        for index, priority in zip(indices, priorities):
            self._priorities[index] = float(priority) + 1e-6

In [6]:
from tensorflow import keras


def build_network(input_shape: tuple, n_action: int):
    inputs = keras.Input(shape=input_shape)

    x = keras.layers.Flatten()(inputs)
    x = keras.layers.Dense(
        units=256,
        activation='relu',
        kernel_initializer='he_normal',
    )(x)
    x = keras.layers.Dense(
        units=256,
        activation='relu',
        kernel_initializer='he_normal',
    )(x)

    value = keras.layers.Dense(
        units=128,
        activation='relu',
        kernel_initializer='he_normal',
    )(x)
    value = keras.layers.Dense(
        units=1,
        activation='linear',
        kernel_initializer='glorot_uniform',
    )(value)

    advantage = keras.layers.Dense(
        units=128,
        activation='relu',
        kernel_initializer='he_normal',
    )(x)
    advantage = keras.layers.Dense(
        units=n_action,
        activation='linear',
        kernel_initializer='glorot_uniform',
    )(advantage)

    mean_advantage = keras.ops.mean(advantage, axis=1, keepdims=True)
    q_values = value + advantage - mean_advantage

    return keras.models.Model(inputs=inputs, outputs=q_values)

In [7]:
import kymnasium as kym
import tensorflow as tf
from tensorflow import keras
from collections import deque
import json
import os


class Agent(kym.Agent):
    def __init__(
            self,
            behavior_network: keras.models.Model,
            target_network: keras.models.Model,
            state_dim: int = STATE_DIM,
            state_seq: int = STATE_SEQ,
            action_space: int = ACTION_DIM,
            buffer_size: int = REPLAY_BUFFER_SIZE,
            alpha: float = ALPHA,
            beta: float = BETA,
            gamma: float = GAMMA,
            tau: float = TAU,
            learning_rate: float = LEARNING_RATE,
            batch_size: int = BATCH_SIZE,
            n_monitors: int = N_MONITOR,
    ):
        self._behavior_network = behavior_network
        self._target_network = target_network
        self._state_seq = state_seq
        self._state_dim = state_dim
        self._action_space = action_space
        self._buffer_size = buffer_size
        self._alpha = alpha
        self._beta = beta
        self._gamma = gamma
        self._tau = tau
        self._learning_rate = learning_rate
        self._batch_size = batch_size
        self._n_monitors = n_monitors

        self._target_network.set_weights(self._behavior_network.get_weights())
        self._replay_buffer = ReplayBuffer(
            capacity=buffer_size,
            alpha=alpha,
            beta=beta
        )
        self._optimizer = keras.optimizers.Adam(learning_rate=self._learning_rate, clipnorm=1.0)
        self._objective = keras.losses.Huber(reduction=None)

        self._losses = deque(maxlen=self._n_monitors)
        self._rewards = deque(maxlen=self._n_monitors)
        self._action_counts = [
            deque(maxlen=self._n_monitors) for _ in range(self._action_space)
        ]

        self._running_state = deque([
            np.zeros(shape=(self._state_dim, )) for _ in range(self._state_seq)
        ], maxlen=self._state_seq)

    @property
    def loss_(self):
        if not len(self._losses):
            return 0.0
        return np.mean(self._losses)

    @property
    def reward_(self):
        if not len(self._rewards):
            return 0.0
        return np.mean(self._rewards)

    @property
    def action_counts_(self):
        return [
            np.mean(self._action_counts[i]) if len(self._action_counts[i]) > 0 else 0.0
            for i in range(self._action_space)
        ]

    def to_state(self, obs):
        mario, blurps = obs['mario'], obs['blurps']
        mario = np.array(
            [mario[0] / WIDTH, mario[1] / HEIGHT, mario[-1] / WIDTH],
            dtype=np.float32
        )
        blurps = np.array([
            [blurp[0] / WIDTH, blurp[1] / HEIGHT, blurp[4] / WIDTH, blurp[5] / HEIGHT, blurp[6] / HEIGHT]
            for blurp in blurps
        ], dtype=np.float32)
        blurps = np.ravel(blurps)

        self._running_state.append(np.concatenate([mario, blurps]).astype(np.float32))
        return np.expand_dims(np.asarray(self._running_state, dtype=np.float32), axis=0)

    def add(self, state, action, reward, next_state, done):
        self._replay_buffer.add(
            state[0], action, reward, next_state[0], done
        )

    def choose_action(self, state):
        state = keras.ops.convert_to_tensor(state)
        q = self._behavior_network(state, training=False)
        q = keras.ops.ravel(q)
        return int(keras.ops.argmax(q).numpy())

    def act(self, obs, info):
        state = self.to_state(obs)
        return self.choose_action(state)

    @tf.function
    def train(self, states, actions, rewards, next_states, dones, weights):
        next_actions = keras.ops.argmax(self._behavior_network(next_states), axis=1)
        next_actions = keras.ops.one_hot(next_actions, self._action_space)
        next_action_values = keras.ops.sum(
            self._target_network(next_states) * next_actions, axis=1, keepdims=True
        )

        targets = rewards + (1 - dones) * self._gamma * next_action_values
        targets = tf.stop_gradient(targets)

        with tf.GradientTape() as tape:
            action_values = keras.ops.sum(
                self._behavior_network(states) * actions, axis=1, keepdims=True
            )
            loss = self._objective(action_values, targets)
            loss = keras.ops.mean(loss * weights)

            priorities = keras.ops.abs(targets - action_values) + 1e-8

        gradients = tape.gradient(loss, self._behavior_network.trainable_variables)
        self._optimizer.apply_gradients(zip(gradients, self._behavior_network.trainable_variables))

        return loss, priorities

    def replay(self):
        if self._replay_buffer.size_ < self._batch_size:
            return

        states, actions, rewards, next_states, dones, indices, weights = self._replay_buffer.sample(self._batch_size)

        loss, priorities = self.train(
            states=keras.ops.convert_to_tensor(states),
            actions=keras.ops.one_hot(actions.astype(np.int32), self._action_space),
            rewards=keras.ops.expand_dims(rewards, axis=1),
            next_states=keras.ops.convert_to_tensor(next_states),
            dones=keras.ops.expand_dims(dones, axis=1),
            weights=keras.ops.expand_dims(weights, axis=1)
        )
        priorities = priorities.numpy().reshape(-1)
        self._replay_buffer.update_priorities(indices, priorities)
        self._losses.append(loss)

        for target_var, behavior_var in zip(
            self._target_network.trainable_variables,
            self._behavior_network.trainable_variables,
        ):
            target_var.assign(
                self._tau * behavior_var + (1.0 - self._tau) * target_var
            )

    def begin_episode(self):
        self._running_state = deque([
            np.zeros(shape=(self._state_dim, )) for _ in range(self._state_seq)
        ], maxlen=self._state_seq)

    def end_episode(self, rewards, actions):
        self._rewards.append(rewards)

        for i in range(self._action_space):
            self._action_counts[i].append(actions[i])

    def save(self, path: str):
        config = dict(
            state_dim=self._state_dim,
            state_seq=self._state_seq,
            action_space=self._action_space,
            buffer_size=self._buffer_size,
            alpha=self._alpha,
            beta=self._beta,
            gamma=self._gamma,
            tau=self._tau,
            learning_rate=self._learning_rate,
            batch_size=self._batch_size,
            n_monitors=self._n_monitors,
        )

        os.makedirs(path, exist_ok=True)
        with open(os.path.join(path, 'config.json'), "w") as file:
            json.dump(config, file)

        keras.models.save_model(self._behavior_network, os.path.join(path, 'behavior_network.keras'))
        keras.models.save_model(self._target_network, os.path.join(path, 'target_network.keras'))

    @classmethod
    def load(cls, path: str) -> 'Agent':
        with open(os.path.join(path, 'config.json')) as file:
            config = json.load(file)
        behavior_network = keras.models.load_model(os.path.join(path, 'behavior_network.keras'))
        target_network = keras.models.load_model(os.path.join(path, 'target_network.keras'))

        return Agent(
            behavior_network = behavior_network,
            target_network = target_network,
            **config
        )

    def set_params(self, **kwargs):
        for key, value in kwargs.items():
            key = f'_{key}'
            if hasattr(self, key):
                setattr(self, key, value)
            else:
                raise AttributeError(f"'Agent' object has no attribute '{key}'")

In [8]:
from tqdm.auto import tqdm


def learn(
        n_episodes: int,
        init_exploration: int,
        eps_decay_steps: int,
        min_epsilon: float,
        save_interval: int,
        update_interval: int,
        agent: Agent,
        random_state: int = None,
):
    pbar = tqdm(range(n_episodes), desc='episode')
    global_steps = 0
    random = np.random.default_rng(random_state)
    epsilon = 1.0

    for episode in pbar:
        steps, total_reward, action_counts = 0, 0.0, np.zeros(3)

        agent.begin_episode()
        done = False
        obs, _ = env.reset()
        state = agent.to_state(obs)

        while not done:
            if global_steps >= init_exploration:
                progress = min(1.0, (global_steps - init_exploration) / eps_decay_steps)
                epsilon = 1.0 + progress * (min_epsilon - 1.0)

            if random.random() < epsilon:
                action = random.choice(3)
            else:
                action = agent.choose_action(state)

            next_obs, _, cleared, dead, _ = env.step(action)
            next_state = agent.to_state(next_obs)

            done = cleared or dead
            if cleared:
                reward = 10.0
            elif dead:
                reward = -10.0
            else:
                reward = 0.01

            agent.add(state, action, reward, next_state, done)

            if steps % update_interval == 0:
                agent.replay()

            state = next_state

            total_reward += reward
            steps += 1
            global_steps += 1
            action_counts[action] += 1

        if action_counts.sum() > 0:
            action_counts = action_counts / action_counts.sum()
        agent.end_episode(total_reward, action_counts)

        if episode % save_interval == 0:
            agent.save(os.path.join('./avoid_blurp', f'Ep #{episode}'))

        a1, a2, a3 = agent.action_counts_
        pbar.set_postfix({
            'loss': f'{agent.loss_:.5f}',
            'reward': f'{agent.reward_:.5f}',
            'action=0': f'{a1:.5f}',
            'action=1': f'{a2:.5f}',
            'action=2': f'{a3:.5f}',
            'epsilon': f'{epsilon:.5f}',
        })

D:\Projects\kymnasium\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [146]:
ddqn_agent = Agent(
    behavior_network=build_network((STATE_SEQ, STATE_DIM), ACTION_DIM),
    target_network=build_network((STATE_SEQ, STATE_DIM), ACTION_DIM),
)

learn(
    n_episodes=1000000,
    init_exploration=50000,
    eps_decay_steps=300000,
    min_epsilon=0.05,
    save_interval=10000,
    update_interval=4,
    agent=ddqn_agent,
    random_state=42
)

episode:   2%|▏         | 16952/1000000 [4:25:25<256:31:58,  1.06it/s, loss=0.01590, reward=-8.52070, action=0=0.93958, action=1=0.03131, action=2=0.02910, epsilon=0.05000]


KeyboardInterrupt: 

In [9]:
import kymnasium as kym


agent = Agent.load('./avoid_blurp/Ep #10000')
env_eval = gym.make(
    'kymnasium/AvoidBlurp-Discrete-Ballistic-Normal-Stage-1',
    render_mode='human'
)

obs, info = env_eval.reset()
state = agent.to_state(obs)
done = False

while not done:
    action = agent.act(obs, info)
    next_obs, reward, terminated, truncated, info = env_eval.step(action)
    done = terminated or truncated
    obs = next_obs

In [2]:
import tensorflow as tf


print("GPUs:", tf.config.list_physical_devices("GPU"))

GPUs: []
